# Chapter 1: The wormlike chain and its projected span

*Code cells: MIT licence (`LICENSE`). Text and figures: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang, 2026).*

In [1]:
import sys, json, math, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'tutorial' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
pd.set_option('display.precision', 8)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
REF = pd.read_csv(ROOT / 'data' / 'wlc_mean_span_reference.csv')
LAM, F = REF['lambda'].to_numpy(), REF['F_continuum'].to_numpy()
Z_M, B_COEF, A_GAUSS = 0.71821164, 0.64680, 4/math.sqrt(3*math.pi)
def check(name, value, expected, tol):
    ok = abs(value - expected) <= tol
    print(('PASS' if ok else 'FAIL') + f'  {name}: {value:.10g}  (expected {expected:.10g}, tol {tol:g})')
    return ok

## The model

The ideal three-dimensional wormlike chain (WLC) is a continuous curve $\mathbf r(s)$, $0\le s\le L$,
with unit tangent $\mathbf t(s)$ and bending energy
$$\frac{\mathcal H}{k_BT}=\frac p2\int_0^L\Big|\frac{\partial\mathbf t}{\partial s}\Big|^2\,ds ,$$
so that $\langle\mathbf t(s)\cdot\mathbf t(s')\rangle=e^{-|s-s'|/p}$. Both ends are free and the initial
tangent is isotropic. The tangent performs Brownian motion on the unit sphere with generator
$\tfrac12\Delta_{S^2}$ in units of $p$.

For a fixed unit vector $\hat{\mathbf e}$, the **projected span** of one conformation and the **mean span** are
$$X_c=\max_s\mathbf r(s)\cdot\hat{\mathbf e}-\min_s\mathbf r(s)\cdot\hat{\mathbf e},\qquad X=\langle X_c\rangle=p\,F(\lambda),\qquad\lambda=L/p .$$
By isotropy $X$ does not depend on $\hat{\mathbf e}$. In reduced units the projected position
$Z=\mathbf r\cdot\hat{\mathbf e}/p$ and projected tangent $U=\mathbf t\cdot\hat{\mathbf e}$ obey
$$dZ=U\,d\tau,\qquad dU=-U\,d\tau+\sqrt{1-U^2}\,dW_\tau ,$$
with $U$ uniform on $[-1,1]$ in equilibrium. The pair $(Z,U)$ is Markovian; $Z$ alone is not.

## A direct simulation

The cleanest check of everything that follows is a direct simulation of chains. The sampler in
`code/sphere_heat_kernel.py` draws each tangent step from a numerically inverted series for the
spherical heat kernel (the exact transition law of the tangent over a step $h$, up to truncation and
interpolation), so the only remaining approximation is the straight-segment interpolation of the
position between nodes. We simulate 4000 chains at $\lambda=1$ with step $h=0.01$ and compare the mean
span, averaged over the three coordinate axes, with the reference value.

In [2]:
from sphere_heat_kernel import heat_inverse, heat_step
def simulate_spans(lam, h, N, seed=1):
    u, cdf, diag = heat_inverse(h)
    rng = np.random.default_rng(seed)
    t = rng.normal(size=(N, 3)); t /= np.linalg.norm(t, axis=1)[:, None]
    pos = np.zeros((N, 3)); mx = pos.copy(); mn = pos.copy()
    for _ in range(int(round(lam / h))):
        tn = heat_step(t, u, cdf, rng)
        pos = pos + 0.5 * (t + tn) * h; t = tn
        mx = np.maximum(mx, pos); mn = np.minimum(mn, pos)
    span = (mx - mn).mean(axis=1)             # average over the three axes within each chain
    endpoint = np.abs(pos).mean(axis=1)       # |R . e| averaged over axes
    return span, endpoint, diag
span, endpoint, diag = simulate_spans(1.0, 0.01, 4000)
F_ref = float(REF.loc[REF['lambda'] == 1.0, 'F_continuum'].iloc[0])
print(f'simulated mean span  F = {span.mean():.5f} +/- {span.std()/np.sqrt(len(span)):.5f}  (1 SE)')
print(f'reference value      F = {F_ref:.6f}')
print(f'mean |R.e|/p           = {endpoint.mean():.5f}   (the span exceeds it by {100*(span.mean()/endpoint.mean()-1):.1f} %)')
print('heat-kernel moment error of the sampler:', diag['max_moment_error'])

simulated mean span  F = 0.45749 +/- 0.00061  (1 SE)
reference value      F = 0.456905
mean |R.e|/p           = 0.42744   (the span exceeds it by 7.0 %)
heat-kernel moment error of the sampler: 3.643096802008472e-08


The span always exceeds the projected end-to-end distance, by about 8 % at $\lambda=1$. The two
quantities separate because the chain can fold back and extend beyond either end. (Chapter 4 derives the
leading excess, $9\lambda^2/256$ for short chains.)

## Rod and coil limits

For a rigid rod the projected length is $L|\cos\theta|$ with $\theta$ uniform on the sphere, so
$X=L\langle|\cos\theta|\rangle=L/2$. For a Gaussian coil the projected position is Brownian motion and
$X=4R_g/\sqrt\pi$. Neither limit describes a chain of intermediate length, which is the subject of the
article:

In [3]:
for lam in (0.01, 1.0, 6.790, 100.0):
    i = np.argmin(np.abs(LAM - lam)); l = LAM[i]
    print(f'lambda={l:8.3g}  F={F[i]:10.5f}   rod L/2={l/2:10.5f}   Gaussian 4 sqrt(lambda/3pi)={A_GAUSS*np.sqrt(l):10.5f}')
lam_cross = 64/(3*math.pi)
print(f'\nthe two limits cross at lambda = 64/(3 pi) = {lam_cross:.4f}, where both give {lam_cross/2:.4f}; the actual value there is', np.interp(lam_cross, LAM, F).round(4))

lambda=    0.01  F=   0.00500   rod L/2=   0.00500   Gaussian 4 sqrt(lambda/3pi)=   0.13029
lambda=       1  F=   0.45691   rod L/2=   0.50000   Gaussian 4 sqrt(lambda/3pi)=   1.30294
lambda=       7  F=   2.25203   rod L/2=   3.50000   Gaussian 4 sqrt(lambda/3pi)=   3.44726
lambda=     100  F=  11.65759   rod L/2=  50.00000   Gaussian 4 sqrt(lambda/3pi)=  13.02940

the two limits cross at lambda = 64/(3 pi) = 6.7906, where both give 3.3953; the actual value there is 2.2024


## Figure 1

`notebooks/fig1.ipynb` draws one illustrative conformation and shows that the span grows only on the
*record* segments of the projected position, where $Z$ exceeds its running maximum or falls below its
running minimum. That observation is the starting point of Chapter 2.